In [ ]:
import os
import warnings
import pandas as pd
import numpy as np

# ML Models & Metrics
from sklearn.ensemble import RandomForestRegressor
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')

data_path = os.path.join('data', 'all_months_features.csv')
df_raw = pd.read_csv(data_path)

In [ ]:
# Chronological sorting & Target shifting (t + 1)
df = df_raw.copy()
df['month_idx'] = df.groupby('Product_Name').cumcount() + 1
df = df.sort_values(by=['Product_Name', 'month_idx']).reset_index(drop=True)

targets_base = ['Min_Price', 'Avg_Price', 'Max_Price']
targets_next = ['Min_Price_next', 'Avg_Price_next', 'Max_Price_next']

for base_col, next_col in zip(targets_base, targets_next):
    df[next_col] = df.groupby('Product_Name')[base_col].shift(-1)

df_clean = df.dropna(subset=targets_next).copy().reset_index(drop=True)

In [ ]:
#  Separate Features and Targets
non_feature_cols = [
    'Product_Name', 'Category', 'Unit', 'unit_canonical', 'month_name',
    'bs_year', 'bs_month'
] + targets_base + targets_next

feature_cols = [col for col in df_clean.columns if col not in non_feature_cols]

X = df_clean[feature_cols].copy().apply(pd.to_numeric, errors='coerce').fillna(0)
y = df_clean[targets_next].copy().fillna(0)

In [ ]:
# Temporal Train/Test Split (Months 1-8 Train, Months 9-10 Test)
train_mask = (df_clean['month_idx'] <= 8).values
test_mask = (df_clean['month_idx'] >= 9).values

X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]

# Find index for Avg_Price_next
avg_idx = targets_next.index('Avg_Price_next')

In [ ]:
# RANDOM FOREST: Average Price Prediction
print("--- RANDOM FOREST (Avg_Price_next) ---")
rf_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model = MultiOutputRegressor(rf_base)
rf_model.fit(X_train, y_train)

y_pred_rf_test = rf_model.predict(X_test)
actual_rf = y_test.iloc[:, avg_idx].values
pred_rf = y_pred_rf_test[:, avg_idx]

rf_mae = mean_absolute_error(actual_rf, pred_rf)
rf_rmse = np.sqrt(mean_squared_error(actual_rf, pred_rf))
rf_r2 = r2_score(actual_rf, pred_rf)
print(f"[RF Test] Avg_Price_next -> MAE: {rf_mae:.2f}, RMSE: {rf_rmse:.2f}, R²: {rf_r2:.3f}")

In [ ]:
# LIGHTGBM: Average Price Prediction

print("\n--- LIGHTGBM (Avg_Price_next) ---")
lgb_base = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.05, num_leaves=31, random_state=42, n_jobs=-1, verbose=-1)
lgb_model = MultiOutputRegressor(lgb_base)
lgb_model.fit(X_train, y_train)

y_pred_lgb_test = lgb_model.predict(X_test)
# Constraint enforcement (Min <= Avg <= Max)
y_pred_lgb_test[:, 0] = np.minimum(y_pred_lgb_test[:, 0], y_pred_lgb_test[:, 1])
y_pred_lgb_test[:, 2] = np.maximum(y_pred_lgb_test[:, 2], y_pred_lgb_test[:, 1])

actual_lgb = y_test.iloc[:, avg_idx].values
pred_lgb = y_pred_lgb_test[:, avg_idx]

lgb_mae = mean_absolute_error(actual_lgb, pred_lgb)
lgb_rmse = np.sqrt(mean_squared_error(actual_lgb, pred_lgb))
lgb_r2 = r2_score(actual_lgb, pred_lgb)
print(f"[LightGBM Test] Avg_Price_next -> MAE: {lgb_mae:.2f}, RMSE: {lgb_rmse:.2f}, R²: {lgb_r2:.3f}")